# 01 — Domain-Adaptive Pre-Training (DAPT)
**Research Objective:** Demonstrate reproducible domain-adaptive masked-language pre-training (DAPT) for our customer-support domain (Taglish). This notebook documents corpus preparation, masking example generation, and the DAPT training recipe. It does not retrain models by default; artifacts in `models/dapt-distilmbert/` are treated as canonical outputs.
## Method Overview
**Why this step exists:** DAPT aligns a general language model to domain-specific lexical and phrase distributions, improving downstream supervised tasks when labeled data is limited.

**Assumptions:** the unlabeled QA CSV (`LazadaQA-Taglish-7k.csv`) is representative and the cleaned corpus (line-delimited text) is suitable for masked-language-model (MLM) training.

In [1]:
# Configuration & Reproducibility Controls (single action)
# Explicit paths, deterministic seed, and runtime flags
ROOT = r"D:\\NLP"
DAPT_TRAIN_FILE = ROOT + "\\dapt_corpus_clean.txt"
DAPT_OUTPUT_DIR = ROOT + "\\models\\dapt-distilmbert"
DEFAULT_SEED = 42
RUN_DAPT = False  # Set to True only if you intend to run full training
GENERATE_MASK_EXAMPLE = True

In [2]:
# Helper utilities (copied/inline from nlp_thesis/utils.py, single action)
import os, json, random, logging
import numpy as np
import torch

def set_seed(seed: int = DEFAULT_SEED, deterministic: bool = False):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_logger(name: str = __name__, level: int = logging.INFO):
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        fmt = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        handler.setFormatter(logging.Formatter(fmt))
        logger.addHandler(handler)
    logger.setLevel(level)
    return logger

logger = get_logger("01_DAPT")

In [3]:
# Data cleaning function (single logical unit - adapted from 01_prepare_dapt.py)
import re

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    t = text.lower()
    t = re.sub(r'<.*?>', '', t)
    t = re.sub(r'http\S+|www\S+|https\S+', '', t)
    t = re.sub(r'\S+@\S+', '', t)
    t = re.sub(r'\d{11}', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

In [4]:
# Compose or inspect the DAPT corpus (single action)
# If file exists, show head; else run local minimal preparation (non-destructive)
import pandas as pd
if os.path.exists(DAPT_TRAIN_FILE):
    print("DAPT corpus found. Sample lines:")
    with open(DAPT_TRAIN_FILE, 'r', encoding='utf-8') as fh:
        for i in range(5):
            l = fh.readline().strip()
            if not l: break
            print(f"{i+1}.", l[:200])
else:
    # Minimal local preparation: read local CSV (if present) and save cleaned lines
    local_csv = ROOT + "\\LazadaQA-Taglish-7k.csv"
    if os.path.exists(local_csv):
        df = pd.read_csv(local_csv)
        text_cols = [c for c in df.columns if any(k in c.lower() for k in ['question','answer','text','body','review','content'])]
        lines = []
        for c in text_cols:
            lines += df[c].dropna().astype(str).map(clean_text).tolist()
        lines = [l for l in lines if len(l.split())>3]
        unique_lines = list(dict.fromkeys(lines))  # preserve order, deduplicate
        os.makedirs(os.path.dirname(DAPT_TRAIN_FILE), exist_ok=True)
        with open(DAPT_TRAIN_FILE, 'w', encoding='utf-8') as fh:
            for l in unique_lines:
                fh.write(l + "\n")
        print(f"Prepared {len(unique_lines)} lines and wrote to {DAPT_TRAIN_FILE}")
    else:
        print("No local CSV found; provide `LazadaQA-Taglish-7k.csv` or adjust path.")

DAPT corpus found. Sample lines:
1. meron po para sa iphone 7
2. bakit sabi buy 1 take 1 tapos yung ibang reviews e isa lng daw dumating?...
3. ok po. sorry for the convenience.
4. wer to buy d ink
5. hi richard, yes, remote controller is included in the package. please refer to the product description to see the complete list of package inclusions. thank you!


### DAPT training recipe (controlled)
A single cell below composes the training steps. It is intentionally set `RUN_DAPT=False` by default. Running it will execute a full Transformers Trainer-based MLM workflow (heavy). The mask-figure generation is separated so figures can be created without a full training run and uses the saved corpus.

In [5]:
# Prepare ML pieces & (optionally) run DAPT training (single logical action)
if RUN_DAPT:
    from transformers import AutoTokenizer, DistilBertForMaskedLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer
    from datasets import load_dataset
    
    set_seed(DEFAULT_SEED)
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")
    datasets = load_dataset("text", data_files={"train": DAPT_TRAIN_FILE})
    tokenized = datasets.map(lambda ex: tokenizer(ex["text"], truncation=True), batched=True, remove_columns=["text"])
    # grouping step (single small function)
    BLOCK_SIZE = 128
    def group_texts(examples):
        concatenated = {k: sum(examples[k], []) for k in examples.keys()}
        total_len = len(concatenated[list(concatenated.keys())[0]])
        total_len = (total_len // BLOCK_SIZE) * BLOCK_SIZE
        result = {k: [t[i:i+BLOCK_SIZE] for i in range(0, total_len, BLOCK_SIZE)] for k,t in concatenated.items()}
        result["labels"] = result["input_ids"].copy()
        return result
    lm_datasets = tokenized.map(group_texts, batched=True)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
    model = DistilBertForMaskedLM.from_pretrained("distilbert-base-multilingual-cased")
    training_args = TrainingArguments(output_dir=DAPT_OUTPUT_DIR, num_train_epochs=3, per_device_train_batch_size=16, save_steps=500, save_total_limit=2, report_to="none")
    trainer = Trainer(model=model, args=training_args, train_dataset=lm_datasets["train"], data_collator=data_collator)
    trainer.train()
    trainer.save_model(DAPT_OUTPUT_DIR)
    tokenizer.save_pretrained(DAPT_OUTPUT_DIR)
    print("DAPT training finished and saved to", DAPT_OUTPUT_DIR)
else:
    print("DAPT training cell skipped. To run, set RUN_DAPT=True and provide GPU or sufficient CPU.")

DAPT training cell skipped. To run, set RUN_DAPT=True and provide GPU or sufficient CPU.


In [6]:
# Masking example generation (single action) — reproduces figure using the corpus
if GENERATE_MASK_EXAMPLE:
    try:
        from transformers import AutoTokenizer, DataCollatorForLanguageModeling
        import matplotlib.pyplot as plt
        tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")
        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
        sample_lines = []
        with open(DAPT_TRAIN_FILE, 'r', encoding='utf-8') as fh:
            for _ in range(3):
                l = fh.readline().strip()
                if not l: break
                sample_lines.append(l)
        if sample_lines:
            enc = tokenizer(sample_lines, return_tensors='pt', padding=True, truncation=True)
            batch = [{k: enc[k][i].tolist() for k in enc.keys()} for i in range(len(sample_lines))]
            collated = data_collator(batch)
            print("Masking Example Tokens (red=[MASK] positions shown as [MASK]):")
            for i, txt in enumerate(sample_lines):
                orig_ids = enc['input_ids'][i].tolist()
                input_ids = collated['input_ids'][i].tolist()
                labels = collated['labels'][i].tolist()
                tokens = []
                for orig, inp, lbl in zip(orig_ids, input_ids, labels):
                    token = tokenizer.convert_ids_to_tokens(orig)
                    if lbl == -100:
                        tokens.append(token)
                    else:
                        if inp == tokenizer.mask_token_id:
                            tokens.append('[MASK]')
                        else:
                            tokens.append(tokenizer.convert_ids_to_tokens(inp))
                print(f"Sample {i+1}: {' '.join(tokens[:120])}")
        else:
            print("No sample lines available for masking example.")
    except Exception as e:
        print("Masking example generation failed:", e)
else:
    print("Masking example generation skipped.")

d:\NLP\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Masking Example Tokens (red=[MASK] positions shown as [MASK]):
Sample 1: [CLS] mer ##on po para sa i ##phone 7 [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
Sample 2: [CLS] bak ##it sa ##bi buy 1 take 1 tapo ##s yu ##ng ibang reviews e isa l ##ng da ##w dum ##ating ? . . . [SEP]
Sample 3: [CLS] ok po . sor ##ry for the con ##veni ##ence . [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [7]:
# Diagnostic timing cell — run to identify slow steps (non-destructive)
# Set DIAG_LOAD_MODEL=True to also time full model load (heavy network + disk)
import time, os, traceback
from pprint import pprint

DIAG_LOAD_MODEL = False  # Change to True only if you want to time model download (heavy)
results = {}

# 1) Imports
t0 = time.time()
try:
    import transformers, datasets, torch
    results['imports_ok'] = True
    results['transformers_version'] = getattr(transformers, '__version__', None)
    results['datasets_version'] = getattr(datasets, '__version__', None)
    results['torch_version'] = getattr(torch, '__version__', None)
except Exception as e:
    results['imports_ok'] = False
    results['imports_error'] = str(e)
results['import_time_s'] = round(time.time() - t0, 3)

# 2) Tokenizer load
t1 = time.time()
try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased", use_fast=True)
    results['tokenizer_ok'] = True
except Exception as e:
    results['tokenizer_ok'] = False
    results['tokenizer_error'] = str(e)
results['tokenizer_time_s'] = round(time.time() - t1, 3)

# 3) Optional: model load (heavy)
if DIAG_LOAD_MODEL:
    t2 = time.time()
    try:
        from transformers import DistilBertForMaskedLM
        model = DistilBertForMaskedLM.from_pretrained("distilbert-base-multilingual-cased")
        results['model_ok'] = True
    except Exception as e:
        results['model_ok'] = False
        results['model_error'] = str(e)
    results['model_time_s'] = round(time.time() - t2, 3)
    # cleanup
    try:
        del model
        import gc
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception:
        pass

# 4) DAPT train file quick read check
results['dapt_file_exists'] = os.path.exists(DAPT_TRAIN_FILE)
if results['dapt_file_exists']:
    t3 = time.time()
    sample = []
    with open(DAPT_TRAIN_FILE, 'r', encoding='utf-8') as fh:
        for i, l in enumerate(fh):
            if i >= 100: break
            if l.strip(): sample.append(l.strip())
    results['sample_lines'] = len(sample)
    results['file_read_time_s'] = round(time.time() - t3, 3)
else:
    results['file_read_time_s'] = None

# 5) Tokenization timing on small subset
results['tokenization_time_s_avg'] = None
if results.get('tokenizer_ok') and results.get('sample_lines', 0) > 0:
    # Warm-up and measure 3 repetitions
    reps = 3
    try:
        t4 = time.time()
        for _ in range(reps):
            _ = tokenizer(sample[:50], truncation=True, padding=True)
        results['tokenization_time_s_avg'] = round((time.time() - t4) / reps, 3)
        results['tokenized_batch_size'] = 50 if len(sample) >= 50 else len(sample)
    except Exception as e:
        results['tokenization_error'] = str(e)

# 6) Heuristics / hints
hints = []
if os.name == 'nt':
    hints.append('On Windows, using datasets.map(..., num_proc>1) may be slower due to process spawn; prefer num_proc=1 for small demos.')
if results.get('tokenizer_time_s') and results['tokenizer_time_s'] > 10:
    hints.append('Tokenizer download was slow — consider pre-downloading with a terminal command or enabling a local cache (HUGGINGFACE_HUB_CACHE).')
if DIAG_LOAD_MODEL and results.get('model_time_s') and results['model_time_s'] > 30:
    hints.append('Model download was heavy; prefer using local saved artifacts for demos or run on a machine with better bandwidth.')

results['hints'] = hints

print('\n=== Diagnostic Results ===')
pprint(results)

print('\nNext steps:')
print('- If tokenizer/model downloads dominate time, consider running in a terminal to prefetch:')
print("  python -c \"from transformers import AutoTokenizer; AutoTokenizer.from_pretrained('distilbert-base-multilingual-cased')\"")
print('- For tokenization map steps, re-run with a small subset and/or num_proc=1 on Windows.')


=== Diagnostic Results ===
{'dapt_file_exists': True,
 'datasets_version': '4.5.0',
 'file_read_time_s': 0.0,
 'hints': ['On Windows, using datasets.map(..., num_proc>1) may be slower due '
           'to process spawn; prefer num_proc=1 for small demos.'],
 'import_time_s': 0.305,
 'imports_ok': True,
 'sample_lines': 100,
 'tokenization_time_s_avg': 0.004,
 'tokenized_batch_size': 50,
 'tokenizer_ok': True,
 'tokenizer_time_s': 2.041,
 'torch_version': '2.5.1+cu124',
 'transformers_version': '4.57.6'}

Next steps:
- If tokenizer/model downloads dominate time, consider running in a terminal to prefetch:
  python -c "from transformers import AutoTokenizer; AutoTokenizer.from_pretrained('distilbert-base-multilingual-cased')"
- For tokenization map steps, re-run with a small subset and/or num_proc=1 on Windows.


## Outputs & Observations
- If DAPT model exists at `models/dapt-distilmbert/`, it can be loaded and used as a base for fine-tuning.  
- The notebook keeps explicit flags to avoid accidental long runs.  
- Assumptions: corpus cleaning heuristics (lowercase, PII removal, >3 words) align with the manuscript (Sec 3.3.3).

### Reproducibility & Limitations
- Deterministic seeds are exposed; hardware differences (GPU vs CPU) may still cause small nondeterministic variations in training.  
- Datasets and models are treated as read-only: do not overwrite `models/dapt-distilmbert/` without explicit consent.